In [50]:
#r "nuget: ScottPlot, 5.*"

Error: ScottPlot version 5.* cannot be added because version 5.0.55 was added previously.

In [51]:
using System;
using System.Diagnostics;
using System.IO;
using System.Linq;
using ScottPlot;

In [52]:
public static double SolveSingleThread(double a, double b, Func<double, double> function, double step)
{
    double result = 0.0;
    for (double x = a; x < b; x += step)
    {
        double xNext = Math.Min(x + step, b);
        result += (function(x) + function(xNext)) * (xNext - x) / 2.0;
    }
    return result;
}

public static double Solve(double a, double b, Func<double, double> function, double step, int threadsnumber)
{
    double result = 0.0;
    object locker = new object();
    double range = b - a;
    double stepSize = range / threadsnumber;

    Parallel.For(0, threadsnumber, new ParallelOptions { MaxDegreeOfParallelism = threadsnumber }, i =>
    {
        double threadStart = a + i * stepSize;
        double threadEnd = (i == threadsnumber - 1) ? b : threadStart + stepSize;
        double localResult = 0.0;

        for (double x = threadStart; x < threadEnd; x += step)
        {
            double next = Math.Min(x + step, threadEnd);
            double nextVal = function(next);
            localResult += (function(x) + nextVal) * (next - x) / 2.0;
        }

        lock (locker)
        {
            result += localResult;
        }
    });

    return result;}

In [53]:
double a = -100, b = 100;
Func<double, double> func = Math.Sin;
double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
double requiredAccuracy = 1e-4;
double reference = -Math.Cos(b) + Math.Cos(a);
double bestStep = steps[steps.Length - 1];

foreach (var step in steps)
{
    double result = SolveSingleThread(a, b, func, step);
    double error = Math.Abs(result - reference);
    if (error <= requiredAccuracy)
    {
        bestStep = step;
        break;
    }
}
display($"\nSelected step: {bestStep}\n");


Selected step: 0,1


In [54]:
int[] threadCounts = { 1, 2, 4, 8, 10, 12, 14, 16 };
int runs = 5;
var results = new System.Collections.Generic.List<(int threads, double avgTime, double result, double error)>();

foreach (var threads in threadCounts)
{
    double sumTime = 0, lastResult = 0;
    for (int i = 0; i < runs; i++)
    {
        var sw = Stopwatch.StartNew();
        lastResult = Solve(a, b, func, bestStep, threads);
        sw.Stop();
        sumTime += sw.Elapsed.TotalMilliseconds;
    }
    double avgTime = sumTime / runs;
    double error = Math.Abs(lastResult - reference);
    results.Add((threads, avgTime, lastResult, error));
}

In [ ]:
double sumSingle = 0, lastSingle = 0;
for (int i = 0; i < runs; i++)
{
    var sw = Stopwatch.StartNew();
    lastSingle = SolveSingleThread(a, b, func, bestStep);
    sw.Stop();
    sumSingle += sw.Elapsed.TotalMilliseconds;
}
double avgSingle = sumSingle / runs;
double errorSingle = Math.Abs(lastSingle - reference);
display($"SingleThread: AvgTime: {avgSingle:F2} ms, Result: {lastSingle}");

SingleThread: AvgTime: 0,05 ms, Result: -2,576362269798923E-15, Error: 2,576362269798923E-15


In [55]:
var bestMulti = results.OrderBy(r => r.avgTime).First();
double percentFaster = 100.0 * (avgSingle - bestMulti.avgTime) / avgSingle;

display($"Лучшее время многопоточной реализации: {bestMulti.avgTime:F2} мс при {bestMulti.threads} потоках");
display($"Время однопоточной реализации: {avgSingle:F2} мс");
display($"Многопоточная версия быстрее на {percentFaster:F1}%");

Лучшее время многопоточной реализации: 0,01 мс при 12 потоках

Время однопоточной реализации: 0,05 мс

Многопоточная версия быстрее на 74,2%

In [56]:
using (var writer = new StreamWriter("experiment_results.txt"))
{
    writer.WriteLine($"Step: {bestStep}");
    writer.WriteLine($"Reference: {reference}");
    writer.WriteLine("Threads\tAvgTime(ms)\tResult\tError");
    foreach (var r in results)
        writer.WriteLine($"{r.threads}\t{r.avgTime:F2}\t{r.result}\t{r.error}");
    writer.WriteLine($"SingleThread\t{avgSingle:F2}\t{lastSingle}\t{errorSingle}");
}

In [58]:
var threadCounts = results.Select(x => (double)x.threads).ToArray();
var avgTimes = results.Select(x => x.avgTime).ToArray();

var scottPlot = new ScottPlot.Plot();
scottPlot.Add.Scatter(threadCounts, avgTimes);
scottPlot.XLabel("Количество потоков");
scottPlot.YLabel("Время выполнения (мс)");
scottPlot.Title("Производительность многопоточного интегрирования");

scottPlot.SavePng("benchmark.png", 800, 600);